# REPATH 02 — Train, evaluate, and ablate the LambdaRank model

This notebook reads the feature dataset generated by REPATH 01 and performs four-fold site-grouped model selection over the 16 development sites. It then trains the final model, evaluates the six held-out test sites, reports gain-based feature importance, and evaluates all 31 non-empty feature-group combinations.

Conditional metrics use only retrievable queries. End-to-end metrics use all framework queries and score retrieval failures as zero. Confidence-margin calibration is performed in REPATH 03.


## 1. Import dependencies


In [ ]:
import json
import warnings
from itertools import combinations
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "configs" / "site_split.yaml").is_file()
        ):
            return candidate
    raise RuntimeError(
        "Run this notebook from the repository root or the notebooks directory."
    )


print("Training imports ready.")


## 2. Configure repository inputs and outputs


In [ ]:
REPO_ROOT = find_repository_root()
FEATURE_ROOT = REPO_ROOT / "generated" / "repath" / "features"
ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "repath"
MODEL_ROOT = REPO_ROOT / "models"
RESULT_ROOT = REPO_ROOT / "results" / "notebooks" / "training"

PAIR_CSV = FEATURE_ROOT / "pairs_real_query_shortlist_50_features.csv"
QUERY_MANIFEST_CSV = ARTIFACT_ROOT / "repath_query_manifest.csv"
FEATURE_SCHEMA_CSV = FEATURE_ROOT / "repath_50_feature_schema.csv"

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
CV_FOLDS = 4
OPTUNA_TRIALS = 200
HPO_MAX_BOOST_ROUNDS = 500
FINAL_MAX_BOOST_ROUNDS = 1000
EARLY_STOPPING_ROUNDS = 50

print("Repository root  :", REPO_ROOT)
print("Pair-feature CSV :", PAIR_CSV)
print("Query manifest   :", QUERY_MANIFEST_CSV)
print("Feature schema   :", FEATURE_SCHEMA_CSV)
print("Training reports :", RESULT_ROOT)
print("Model output     :", MODEL_ROOT)
print("Calibration data :", ARTIFACT_ROOT)
print("CV folds         :", CV_FOLDS)
print("Optuna trials    :", OPTUNA_TRIALS)


## 3. Load the ordered 50-feature schema


In [ ]:
feature_schema_df = pd.read_csv(
    FEATURE_SCHEMA_CSV,
    keep_default_na=False,
)

required_schema_columns = {
    "vector_index", "feature", "group", "dtype", "semantic_model"
}
missing_schema_columns = sorted(
    required_schema_columns - set(feature_schema_df.columns)
)
if missing_schema_columns:
    raise ValueError(
        f"Feature schema is missing columns: {missing_schema_columns}"
    )

feature_schema_df["vector_index"] = pd.to_numeric(
    feature_schema_df["vector_index"],
    errors="raise",
).astype(int)
feature_schema_df = feature_schema_df.sort_values(
    "vector_index",
    kind="mergesort",
).reset_index(drop=True)

if feature_schema_df["vector_index"].tolist() != list(range(50)):
    raise ValueError("Feature schema must contain vector indices 0 through 49")

if feature_schema_df["feature"].duplicated().any():
    duplicates = feature_schema_df.loc[
        feature_schema_df["feature"].duplicated(),
        "feature",
    ].tolist()
    raise ValueError(f"Duplicate feature names: {duplicates}")

expected_groups = [
    "lexical", "semantic", "structural", "attribute", "geometry"
]
observed_groups = list(dict.fromkeys(feature_schema_df["group"].tolist()))
if observed_groups != expected_groups:
    raise ValueError(
        f"Expected ordered groups {expected_groups}, got {observed_groups}"
    )

ALL_FEATURES = feature_schema_df["feature"].tolist()
LEXICAL_FEATURES = feature_schema_df.loc[
    feature_schema_df["group"] == "lexical", "feature"
].tolist()
SEMANTIC_FEATURES = feature_schema_df.loc[
    feature_schema_df["group"] == "semantic", "feature"
].tolist()
STRUCTURAL_FEATURES = feature_schema_df.loc[
    feature_schema_df["group"] == "structural", "feature"
].tolist()
ATTRIBUTE_FEATURES = feature_schema_df.loc[
    feature_schema_df["group"] == "attribute", "feature"
].tolist()
GEOMETRY_FEATURES = feature_schema_df.loc[
    feature_schema_df["group"] == "geometry", "feature"
].tolist()

expected_counts = {
    "lexical": 12,
    "semantic": 5,
    "structural": 8,
    "attribute": 20,
    "geometry": 5,
}
actual_counts = (
    feature_schema_df["group"]
    .value_counts()
    .to_dict()
)
if actual_counts != expected_counts:
    raise ValueError(
        f"Unexpected feature-group counts: {actual_counts}"
    )

semantic_models = [
    value
    for value in feature_schema_df.loc[
        feature_schema_df["group"] == "semantic",
        "semantic_model",
    ].unique().tolist()
    if value
]
if len(semantic_models) != 1:
    raise ValueError(
        f"Expected one semantic model in the schema, got {semantic_models}"
    )
SBERT_MODEL_NAME = semantic_models[0]

assert len(ALL_FEATURES) == 50
assert len(ALL_FEATURES) == len(set(ALL_FEATURES))

print("Feature schema loaded")
print("  Lexical    :", len(LEXICAL_FEATURES))
print("  Semantic   :", len(SEMANTIC_FEATURES))
print("  Structural :", len(STRUCTURAL_FEATURES))
print("  Attribute  :", len(ATTRIBUTE_FEATURES))
print("  Geometry   :", len(GEOMETRY_FEATURES))
print("  Total      :", len(ALL_FEATURES))
print("  SBERT model:", SBERT_MODEL_NAME)
display(feature_schema_df)


## 4. Load and validate the feature dataset


In [ ]:
def parse_bool_series(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )


pairs_df = pd.read_csv(PAIR_CSV, keep_default_na=False)
manifest_df = pd.read_csv(QUERY_MANIFEST_CSV, keep_default_na=False)

required_pair_columns = {
    "qid", "site", "split", "label", "relevance",
    "n_candidates",
    *ALL_FEATURES,
}
missing_pair_columns = sorted(
    required_pair_columns - set(pairs_df.columns)
)
if missing_pair_columns:
    raise ValueError(
        f"Pair CSV is missing columns: {missing_pair_columns}"
    )

required_manifest_columns = {
    "qid", "site", "split", "positive_retained",
    "written_to_pair_csv", "candidate_count", "reject_reason",
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise ValueError(
        f"Query manifest is missing columns: {missing_manifest_columns}"
    )

for frame in (pairs_df, manifest_df):
    frame["qid"] = frame["qid"].astype(str)
    frame["site"] = frame["site"].astype(str)
    frame["split"] = (
        frame["split"].astype(str).str.strip().str.lower()
    )

pairs_df["label"] = pd.to_numeric(
    pairs_df["label"], errors="raise"
).astype(int)
pairs_df["relevance"] = pd.to_numeric(
    pairs_df["relevance"], errors="raise"
).astype(int)
pairs_df["n_candidates"] = pd.to_numeric(
    pairs_df["n_candidates"], errors="raise"
).astype(int)

manifest_df["positive_retained"] = parse_bool_series(
    manifest_df["positive_retained"]
)
manifest_df["written_to_pair_csv"] = parse_bool_series(
    manifest_df["written_to_pair_csv"]
)
manifest_df["candidate_count"] = pd.to_numeric(
    manifest_df["candidate_count"], errors="raise"
).astype(int)

valid_splits = {"development", "test"}
for name, frame in [("pairs", pairs_df), ("manifest", manifest_df)]:
    invalid_splits = sorted(set(frame["split"]) - valid_splits)
    if invalid_splits:
        raise ValueError(
            f"{name} contains invalid splits: {invalid_splits}"
        )

if manifest_df["qid"].duplicated().any():
    raise ValueError("The query manifest must contain one row per qid")

positive_counts = pairs_df.groupby(
    "qid", sort=False
)["label"].sum()
bad_positive_groups = positive_counts[
    positive_counts != 1
]
if len(bad_positive_groups):
    raise ValueError(
        f"{len(bad_positive_groups)} pair groups do not contain "
        f"exactly one positive"
    )

positive_relevance = pairs_df.loc[
    pairs_df["label"] == 1, "relevance"
]
negative_relevance = pairs_df.loc[
    pairs_df["label"] == 0, "relevance"
]
if not positive_relevance.eq(3).all():
    raise ValueError("Every positive row must have relevance 3")
if not negative_relevance.eq(0).all():
    raise ValueError("Every negative row must have relevance 0")

actual_group_sizes = pairs_df.groupby(
    "qid", sort=False
)["qid"].transform("size").astype(int)
if not np.array_equal(
    pairs_df["n_candidates"].to_numpy(),
    actual_group_sizes.to_numpy(),
):
    raise ValueError(
        "n_candidates does not match the actual query-group size"
    )

for feature in ALL_FEATURES:
    pairs_df[feature] = pd.to_numeric(
        pairs_df[feature],
        errors="raise",
    ).astype(np.float32)

feature_matrix = pairs_df[ALL_FEATURES].to_numpy(
    dtype=np.float32
)
if not np.isfinite(feature_matrix).all():
    raise ValueError(
        "The exported 50-feature matrix contains NaN or infinity"
    )

for column in ["split", "site"]:
    if (
        pairs_df.groupby("qid", sort=False)[column]
        .nunique()
        .ne(1)
        .any()
    ):
        raise ValueError(
            f"At least one pair qid appears under multiple {column}s"
        )

pair_qids = set(pairs_df["qid"])
manifest_qids = set(manifest_df["qid"])
retained_manifest_qids = set(
    manifest_df.loc[
        manifest_df["positive_retained"], "qid"
    ]
)
written_manifest_qids = set(
    manifest_df.loc[
        manifest_df["written_to_pair_csv"], "qid"
    ]
)

if not pair_qids.issubset(manifest_qids):
    raise ValueError(
        "Pair CSV contains qids absent from the query manifest"
    )
if pair_qids != retained_manifest_qids:
    raise ValueError(
        "Pair qids do not exactly match positive-retained manifest qids"
    )
if pair_qids != written_manifest_qids:
    raise ValueError(
        "Pair qids do not exactly match written manifest qids"
    )

pair_identity = (
    pairs_df[["qid", "site", "split"]]
    .drop_duplicates("qid")
    .set_index("qid")
)
manifest_identity = (
    manifest_df[["qid", "site", "split"]]
    .set_index("qid")
)
common_qids = pair_identity.index.intersection(
    manifest_identity.index
)
if not pair_identity.loc[common_qids].equals(
    manifest_identity.loc[common_qids]
):
    raise ValueError(
        "Pair and manifest site/split identities do not match"
    )

development_sites = set(
    manifest_df.loc[
        manifest_df["split"] == "development", "site"
    ]
)
test_sites = set(
    manifest_df.loc[
        manifest_df["split"] == "test", "site"
    ]
)

if development_sites & test_sites:
    raise ValueError("Development/test site leakage")
if len(development_sites) != 16:
    raise ValueError(
        f"Expected 16 development sites, found {len(development_sites)}"
    )
if len(test_sites) != 6:
    raise ValueError(
        f"Expected 6 test sites, found {len(test_sites)}"
    )

split_overview = []
for split in ["development", "test"]:
    manifest_split = manifest_df[
        manifest_df["split"] == split
    ]
    pair_split = pairs_df[pairs_df["split"] == split]
    split_overview.append({
        "split": split,
        "framework_queries": manifest_split["qid"].nunique(),
        "retrievable_queries": pair_split["qid"].nunique(),
        "missing_positive_queries": (
            manifest_split["qid"].nunique()
            - pair_split["qid"].nunique()
        ),
        "candidate_recall": (
            pair_split["qid"].nunique()
            / manifest_split["qid"].nunique()
            if manifest_split["qid"].nunique() else np.nan
        ),
        "pair_rows": len(pair_split),
        "sites": manifest_split["site"].nunique(),
    })

split_overview_df = pd.DataFrame(split_overview)

print("CSV-only dataset integrity checks passed")
print("Pair rows        :", f"{len(pairs_df):,}")
print("Framework queries:", f"{manifest_df['qid'].nunique():,}")
print("Retrievable      :", f"{pairs_df['qid'].nunique():,}")
display(split_overview_df)

feature_audit_df = pd.DataFrame({
    "feature": ALL_FEATURES,
    "group": [
        feature_schema_df.set_index("feature").loc[
            feature, "group"
        ]
        for feature in ALL_FEATURES
    ],
    "min": [
        float(pairs_df[feature].min())
        for feature in ALL_FEATURES
    ],
    "max": [
        float(pairs_df[feature].max())
        for feature in ALL_FEATURES
    ],
    "non_zero_fraction": [
        float((pairs_df[feature] != 0).mean())
        for feature in ALL_FEATURES
    ],
    "unique_values": [
        int(pairs_df[feature].nunique())
        for feature in ALL_FEATURES
    ],
})
feature_audit_df["constant"] = (
    feature_audit_df["unique_values"] <= 1
)
display(feature_audit_df)


## 5. Create deterministic site-grouped folds

The 16 development websites are assigned to four folds. Each fold contains
four whole websites. Assignment is deterministic and balances framework
query counts without using model predictions or accuracy.


In [ ]:
def sort_ranking_rows(frame):
    sort_columns = ["qid"]
    if "cand_new_idx" in frame.columns:
        sort_columns.append("cand_new_idx")
    return frame.sort_values(
        sort_columns,
        kind="mergesort",
    ).reset_index(drop=True)


development_df = sort_ranking_rows(
    pairs_df[pairs_df["split"] == "development"].copy()
)
test_df = sort_ranking_rows(
    pairs_df[pairs_df["split"] == "test"].copy()
)


def ranking_group_sizes(frame):
    group_sizes = (
        frame.groupby("qid", sort=False)
        .size()
        .to_numpy(dtype=np.int32)
    )
    if int(group_sizes.sum()) != len(frame):
        raise ValueError("Ranking-group sizes do not sum to row count")
    return group_sizes


def make_lgb_dataset(
    frame,
    features,
    *,
    reference=None,
):
    frame = sort_ranking_rows(frame)
    dataset = lgb.Dataset(
        frame[features].astype(np.float32),
        label=frame["relevance"].to_numpy(dtype=np.int32),
        group=ranking_group_sizes(frame),
        feature_name=features,
        reference=reference,
        free_raw_data=False,
    )
    return dataset, frame


# Deterministic site assignment:
# largest sites are assigned first to the currently lightest fold,
# while every fold is capped at exactly four websites.
development_manifest = manifest_df[
    manifest_df["split"] == "development"
].copy()

site_query_counts = (
    development_manifest.groupby("site")["qid"]
    .nunique()
    .sort_values(ascending=False)
)

development_site_list = sorted(
    site_query_counts.index.tolist(),
    key=lambda site: (
        -int(site_query_counts.loc[site]),
        site.lower(),
    ),
)

if len(development_site_list) % CV_FOLDS != 0:
    raise ValueError(
        "Development site count must be divisible by CV_FOLDS"
    )

sites_per_fold = len(development_site_list) // CV_FOLDS
fold_loads = {fold: 0 for fold in range(CV_FOLDS)}
fold_sites = {fold: [] for fold in range(CV_FOLDS)}

for site in development_site_list:
    eligible_folds = [
        fold
        for fold in range(CV_FOLDS)
        if len(fold_sites[fold]) < sites_per_fold
    ]
    selected_fold = min(
        eligible_folds,
        key=lambda fold: (
            fold_loads[fold],
            len(fold_sites[fold]),
            fold,
        ),
    )
    fold_sites[selected_fold].append(site)
    fold_loads[selected_fold] += int(
        site_query_counts.loc[site]
    )

site_to_fold = {
    site: fold
    for fold, sites in fold_sites.items()
    for site in sites
}

if set(site_to_fold) != set(development_site_list):
    raise ValueError("Not every development site received a fold")
if any(
    len(sites) != sites_per_fold
    for sites in fold_sites.values()
):
    raise ValueError("Every fold must contain the same site count")

manifest_df["cv_fold"] = (
    manifest_df["site"].map(site_to_fold)
    .fillna(-1)
    .astype(int)
)
pairs_df["cv_fold"] = (
    pairs_df["site"].map(site_to_fold)
    .fillna(-1)
    .astype(int)
)
development_df["cv_fold"] = (
    development_df["site"].map(site_to_fold).astype(int)
)
test_df["cv_fold"] = -1

if (development_df["cv_fold"] < 0).any():
    raise ValueError("A development row is missing a CV fold")
if (test_df["cv_fold"] != -1).any():
    raise ValueError("A test row was incorrectly assigned to a CV fold")

cv_fold_rows = []
for fold in range(CV_FOLDS):
    fold_manifest = manifest_df[
        (manifest_df["split"] == "development")
        & (manifest_df["cv_fold"] == fold)
    ]
    fold_pairs = pairs_df[
        (pairs_df["split"] == "development")
        & (pairs_df["cv_fold"] == fold)
    ]

    cv_fold_rows.append({
        "cv_fold": fold,
        "sites": ", ".join(sorted(fold_sites[fold])),
        "site_count": len(fold_sites[fold]),
        "framework_queries": fold_manifest["qid"].nunique(),
        "retrievable_queries": fold_pairs["qid"].nunique(),
        "missing_positive_queries": (
            fold_manifest["qid"].nunique()
            - fold_pairs["qid"].nunique()
        ),
        "candidate_recall": (
            fold_pairs["qid"].nunique()
            / fold_manifest["qid"].nunique()
            if fold_manifest["qid"].nunique() else np.nan
        ),
        "pair_rows": len(fold_pairs),
    })

cv_fold_assignment_df = pd.DataFrame(cv_fold_rows)
display(cv_fold_assignment_df)

site_fold_df = pd.DataFrame([
    {
        "site": site,
        "cv_fold": fold,
        "framework_queries": int(site_query_counts.loc[site]),
    }
    for site, fold in sorted(
        site_to_fold.items(),
        key=lambda item: (item[1], item[0].lower()),
    )
])
site_fold_df.to_csv(
    RESULT_ROOT / "development_site_cv_folds.csv",
    index=False,
)


print("Development/test matrices created from exported CSV features")
print(
    f"Development: {development_df['qid'].nunique():,} groups / "
    f"{len(development_df):,} rows / "
    f"{development_df['site'].nunique()} sites"
)
print(
    f"Test       : {test_df['qid'].nunique():,} groups / "
    f"{len(test_df):,} rows / "
    f"{test_df['site'].nunique()} sites"
)
print("Saved site folds:", RESULT_ROOT / "development_site_cv_folds.csv")


## 6. Select hyperparameters with site-grouped cross-validation

Each trial trains four models. The objective remains mean conditional NDCG@5 because LambdaRank is fitted only to retrievable ranking groups. Nevertheless, both conditional and end-to-end fold metrics are recorded for every trial.


In [ ]:
def dcg_at_k(labels, k):
    labels = np.asarray(labels, dtype=float)[:k]
    if len(labels) == 0:
        return 0.0
    gains = np.power(2.0, labels) - 1.0
    discounts = np.log2(np.arange(2, len(labels) + 2))
    return float(np.sum(gains / discounts))


def ndcg_at_k(labels, k):
    labels = np.asarray(labels, dtype=float)
    ideal_dcg = dcg_at_k(np.sort(labels)[::-1], k)
    return (
        dcg_at_k(labels, k) / ideal_dcg
        if ideal_dcg > 0
        else 0.0
    )


def ranking_query_details(scored_df):
    rows = []

    for qid, group in scored_df.groupby("qid", sort=False):
        ranked = group.sort_values(
            "score",
            ascending=False,
            kind="mergesort",
        ).reset_index(drop=True)

        positive_positions = np.flatnonzero(
            ranked["label"].to_numpy() == 1
        )
        if len(positive_positions) != 1:
            raise ValueError(
                f"{qid}: expected exactly one positive, "
                f"found {len(positive_positions)}"
            )

        positive_rank = int(positive_positions[0] + 1)
        relevance = ranked["relevance"].to_numpy(dtype=float)

        row = {
            "qid": qid,
            "positive_rank": positive_rank,
            "reciprocal_rank": 1.0 / positive_rank,
            "top1": int(positive_rank == 1),
            "hit_at_3": int(positive_rank <= 3),
            "hit_at_5": int(positive_rank <= 5),
            "ndcg_at_1": ndcg_at_k(relevance, 1),
            "ndcg_at_3": ndcg_at_k(relevance, 3),
            "ndcg_at_5": ndcg_at_k(relevance, 5),
            "candidates": int(len(ranked)),
        }

        if "site" in ranked.columns:
            row["site"] = ranked.iloc[0]["site"]
        if "cv_fold" in ranked.columns:
            row["cv_fold"] = int(ranked.iloc[0]["cv_fold"])

        rows.append(row)

    return pd.DataFrame(rows)


def ranking_metrics_from_details(details):
    if details.empty:
        return {
            "queries": 0,
            "top1": np.nan,
            "mrr": np.nan,
            "hit3": np.nan,
            "hit5": np.nan,
            "ndcg1": np.nan,
            "ndcg3": np.nan,
            "ndcg5": np.nan,
            "mean_rank": np.nan,
            "median_rank": np.nan,
        }

    return {
        "queries": int(len(details)),
        "top1": float(details["top1"].mean()),
        "mrr": float(details["reciprocal_rank"].mean()),
        "hit3": float(details["hit_at_3"].mean()),
        "hit5": float(details["hit_at_5"].mean()),
        "ndcg1": float(details["ndcg_at_1"].mean()),
        "ndcg3": float(details["ndcg_at_3"].mean()),
        "ndcg5": float(details["ndcg_at_5"].mean()),
        "mean_rank": float(details["positive_rank"].mean()),
        "median_rank": float(details["positive_rank"].median()),
    }


def ranking_metrics(scored_df):
    return ranking_metrics_from_details(
        ranking_query_details(scored_df)
    )


def end_to_end_metrics(
    conditional_details,
    split,
    name,
    *,
    cv_fold=None,
    site=None,
):
    framework_mask = manifest_df["split"] == split

    if cv_fold is not None:
        framework_mask &= (
            manifest_df["cv_fold"] == int(cv_fold)
        )

    if site is not None:
        framework_mask &= (
            manifest_df["site"] == site
        )

    framework = (
        manifest_df.loc[
            framework_mask,
            ["qid", "site"],
        ]
        .drop_duplicates("qid")
        .copy()
    )

    details = framework.merge(
        conditional_details,
        on=["qid", "site"],
        how="left",
        validate="one_to_one",
    )

    metric_columns = [
        "reciprocal_rank",
        "top1",
        "hit_at_3",
        "hit_at_5",
        "ndcg_at_1",
        "ndcg_at_3",
        "ndcg_at_5",
    ]
    for column in metric_columns:
        details[column] = pd.to_numeric(
            details[column],
            errors="coerce",
        ).fillna(0.0)

    details["retrieval_failure"] = (
        details["positive_rank"].isna()
    ).astype(int)

    details["candidates"] = pd.to_numeric(
        details["candidates"],
        errors="coerce",
    ).fillna(0).astype(int)

    metrics = {
        "name": name,
        "framework_queries": int(len(details)),
        "retrievable_queries": int(
            (details["retrieval_failure"] == 0).sum()
        ),
        "retrieval_failures": int(
            details["retrieval_failure"].sum()
        ),
        "candidate_recall": float(
            (details["retrieval_failure"] == 0).mean()
        ),
        "top1": float(details["top1"].mean()),
        "mrr": float(details["reciprocal_rank"].mean()),
        "hit3": float(details["hit_at_3"].mean()),
        "hit5": float(details["hit_at_5"].mean()),
        "ndcg1": float(details["ndcg_at_1"].mean()),
        "ndcg3": float(details["ndcg_at_3"].mean()),
        "ndcg5": float(details["ndcg_at_5"].mean()),
    }

    return metrics, details


def evaluate_scored_predictions(
    scored_df,
    split,
    name,
    *,
    cv_fold=None,
    site=None,
):
    conditional_details = ranking_query_details(scored_df)
    conditional = ranking_metrics_from_details(
        conditional_details
    )
    end_to_end, end_to_end_details = end_to_end_metrics(
        conditional_details,
        split,
        f"{name} end-to-end",
        cv_fold=cv_fold,
        site=site,
    )

    combined = {
        "name": name,
        "conditional_queries": conditional["queries"],
        "conditional_top1": conditional["top1"],
        "conditional_mrr": conditional["mrr"],
        "conditional_hit3": conditional["hit3"],
        "conditional_hit5": conditional["hit5"],
        "conditional_ndcg1": conditional["ndcg1"],
        "conditional_ndcg3": conditional["ndcg3"],
        "conditional_ndcg5": conditional["ndcg5"],
        "conditional_mean_rank": conditional["mean_rank"],
        "conditional_median_rank": conditional["median_rank"],
        "framework_queries": end_to_end["framework_queries"],
        "retrievable_queries": end_to_end[
            "retrievable_queries"
        ],
        "retrieval_failures": end_to_end[
            "retrieval_failures"
        ],
        "candidate_recall": end_to_end["candidate_recall"],
        "end_to_end_top1": end_to_end["top1"],
        "end_to_end_mrr": end_to_end["mrr"],
        "end_to_end_hit3": end_to_end["hit3"],
        "end_to_end_hit5": end_to_end["hit5"],
        "end_to_end_ndcg1": end_to_end["ndcg1"],
        "end_to_end_ndcg3": end_to_end["ndcg3"],
        "end_to_end_ndcg5": end_to_end["ndcg5"],
    }

    return (
        combined,
        conditional,
        end_to_end,
        conditional_details,
        end_to_end_details,
    )


def effective_iteration(model_local):
    best = getattr(model_local, "best_iteration", None)
    return (
        int(best)
        if best is not None and best > 0
        else int(model_local.current_iteration())
    )


def common_lgb_params():
    return {
        "objective": "lambdarank",
        "metric": "ndcg",
        "ndcg_eval_at": [5],
        "verbosity": -1,
        "seed": SEED,
        "random_state": SEED,
        "feature_fraction_seed": SEED,
        "bagging_seed": SEED,
        "data_random_seed": SEED,
        "drop_seed": SEED,
        "deterministic": True,
        "force_col_wise": True,
        "num_threads": 1,
        "bagging_freq": 1,
        "label_gain": [0, 1, 3, 7],
        "feature_pre_filter": False,
    }


def params_from_trial(trial):
    params = common_lgb_params()
    params.update({
        "learning_rate": trial.suggest_float(
            "lr", 0.005, 0.05, log=True
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves", 12, 64
        ),
        "min_data_in_leaf": trial.suggest_int(
            "min_data", 10, 60
        ),
        "feature_fraction": trial.suggest_float(
            "feat_frac", 0.55, 1.0
        ),
        "bagging_fraction": trial.suggest_float(
            "bag_frac", 0.65, 1.0
        ),
        "lambda_l1": trial.suggest_float(
            "l1", 0.0, 2.0
        ),
        "lambda_l2": trial.suggest_float(
            "l2", 0.0, 2.0
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 3, 8
        ),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_hess", 1e-3, 10.0, log=True
        ),
    })
    return params


def fit_fold_with_early_stopping(
    params,
    fold,
    features,
    *,
    max_rounds,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
):
    fold_train = development_df[
        development_df["cv_fold"] != fold
    ].copy()
    fold_val = development_df[
        development_df["cv_fold"] == fold
    ].copy()

    train_set, fold_train = make_lgb_dataset(
        fold_train,
        features,
    )
    val_set, fold_val = make_lgb_dataset(
        fold_val,
        features,
        reference=train_set,
    )

    fold_model = lgb.train(
        params,
        train_set,
        valid_sets=[val_set],
        valid_names=[f"fold_{fold}_validation"],
        num_boost_round=max_rounds,
        callbacks=[
            lgb.early_stopping(
                early_stopping_rounds,
                verbose=False,
            ),
            lgb.log_evaluation(0),
        ],
    )

    scored = fold_val[
        ["qid", "site", "cv_fold", "label", "relevance"]
    ].copy().reset_index(drop=True)
    scored["score"] = fold_model.predict(
        fold_val[features].astype(np.float32),
        num_iteration=effective_iteration(fold_model),
    )

    metrics, _, _, _, _ = evaluate_scored_predictions(
        scored,
        "development",
        f"Development fold {fold}",
        cv_fold=fold,
    )

    return fold_model, metrics, scored


def fit_fold_fixed_rounds(
    params,
    fold,
    features,
    num_boost_round,
):
    fold_train = development_df[
        development_df["cv_fold"] != fold
    ].copy()
    fold_val = sort_ranking_rows(
        development_df[
            development_df["cv_fold"] == fold
        ].copy()
    )

    train_set, fold_train = make_lgb_dataset(
        fold_train,
        features,
    )

    fold_model = lgb.train(
        params,
        train_set,
        num_boost_round=int(num_boost_round),
        callbacks=[lgb.log_evaluation(0)],
    )

    scored = fold_val[
        ["qid", "site", "cv_fold", "label", "relevance"]
    ].copy().reset_index(drop=True)
    scored["score"] = fold_model.predict(
        fold_val[features].astype(np.float32),
        num_iteration=int(num_boost_round),
    )

    metrics, _, _, _, _ = evaluate_scored_predictions(
        scored,
        "development",
        f"Development fold {fold}",
        cv_fold=fold,
    )

    return fold_model, metrics, scored


def fit_full_development_model(
    params,
    features,
    num_boost_round,
):
    development_set, _ = make_lgb_dataset(
        development_df,
        features,
    )

    return lgb.train(
        params,
        development_set,
        num_boost_round=int(num_boost_round),
        callbacks=[lgb.log_evaluation(0)],
    )


HPO_METRICS = [
    "conditional_top1",
    "conditional_mrr",
    "conditional_hit3",
    "conditional_hit5",
    "conditional_ndcg1",
    "conditional_ndcg3",
    "conditional_ndcg5",
    "candidate_recall",
    "end_to_end_top1",
    "end_to_end_mrr",
    "end_to_end_hit3",
    "end_to_end_hit5",
    "end_to_end_ndcg1",
    "end_to_end_ndcg3",
    "end_to_end_ndcg5",
]


def optuna_objective(trial):
    params = params_from_trial(trial)
    fold_metrics = []

    for fold in range(CV_FOLDS):
        _, metrics, _ = fit_fold_with_early_stopping(
            params,
            fold,
            ALL_FEATURES,
            max_rounds=HPO_MAX_BOOST_ROUNDS,
        )
        fold_metrics.append(metrics)

    for metric_name in HPO_METRICS:
        values = np.asarray(
            [
                metrics[metric_name]
                for metrics in fold_metrics
            ],
            dtype=float,
        )
        trial.set_user_attr(
            f"mean_{metric_name}",
            float(np.nanmean(values)),
        )
        trial.set_user_attr(
            f"std_{metric_name}",
            float(np.nanstd(values, ddof=0)),
        )

    return float(
        np.mean([
            metrics["conditional_ndcg5"]
            for metrics in fold_metrics
        ])
    )


print(
    f"Starting {CV_FOLDS}-fold site-grouped Optuna HPO "
    f"({OPTUNA_TRIALS} trials; "
    f"{OPTUNA_TRIALS * CV_FOLDS} fold fits)..."
)

sampler = optuna.samplers.TPESampler(
    seed=SEED,
    multivariate=True,
    group=True,
)
study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
)
study.optimize(
    optuna_objective,
    n_trials=OPTUNA_TRIALS,
    show_progress_bar=True,
)

best_trial = study.best_trial
best_params = common_lgb_params()
best_params.update({
    "learning_rate": study.best_params["lr"],
    "num_leaves": study.best_params["num_leaves"],
    "min_data_in_leaf": study.best_params["min_data"],
    "feature_fraction": study.best_params["feat_frac"],
    "bagging_fraction": study.best_params["bag_frac"],
    "lambda_l1": study.best_params["l1"],
    "lambda_l2": study.best_params["l2"],
    "max_depth": study.best_params["max_depth"],
    "min_sum_hessian_in_leaf": study.best_params["min_hess"],
})

print("\nBest mean CV conditional NDCG@5:", study.best_value)

for metric_name in HPO_METRICS:
    print(
        f"Best CV {metric_name}: "
        f"{best_trial.user_attrs.get(f'mean_{metric_name}')} "
        f"± "
        f"{best_trial.user_attrs.get(f'std_{metric_name}')}"
    )

print("Best params:", best_params)

study.trials_dataframe().to_csv(
    RESULT_ROOT / "optuna_site_grouped_cv_trials.csv",
    index=False,
)
print(
    "Saved Optuna trials:",
    RESULT_ROOT / "optuna_site_grouped_cv_trials.csv",
)


## 7. Select boosting rounds and train the final model


In [ ]:
# Cross-validated boosting-round selection

round_selection_rows = []

for fold in range(CV_FOLDS):
    fold_model_es, fold_metrics_es, _ = (
        fit_fold_with_early_stopping(
            best_params,
            fold,
            ALL_FEATURES,
            max_rounds=FINAL_MAX_BOOST_ROUNDS,
        )
    )

    round_selection_rows.append({
        "cv_fold": fold,
        "best_iteration": effective_iteration(
            fold_model_es
        ),
        **{
            key: value
            for key, value in fold_metrics_es.items()
            if key != "name"
        },
    })

cv_round_selection_df = (
    pd.DataFrame(round_selection_rows)
    .sort_values("cv_fold")
    .reset_index(drop=True)
)

FINAL_NUM_BOOST_ROUND = max(
    1,
    int(round(
        float(
            cv_round_selection_df[
                "best_iteration"
            ].median()
        )
    )),
)

print("Cross-validated early-stopping results:")
display(cv_round_selection_df)

print(
    "Selected common boosting rounds:",
    FINAL_NUM_BOOST_ROUND,
)


# Fixed-round out-of-fold development predictions

oof_parts = []
cv_fixed_round_rows = []

for fold in range(CV_FOLDS):
    _, fold_metrics, fold_scored = fit_fold_fixed_rounds(
        best_params,
        fold,
        ALL_FEATURES,
        FINAL_NUM_BOOST_ROUND,
    )

    oof_parts.append(fold_scored)
    cv_fixed_round_rows.append({
        "cv_fold": fold,
        "features": len(ALL_FEATURES),
        "num_boost_round": FINAL_NUM_BOOST_ROUND,
        **{
            key: value
            for key, value in fold_metrics.items()
            if key != "name"
        },
    })

oof_scored = (
    pd.concat(oof_parts, ignore_index=True)
    .sort_values(
        ["qid", "score"],
        ascending=[True, False],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

expected_oof_qids = set(
    development_df["qid"].astype(str)
)
observed_oof_qids = set(
    oof_scored["qid"].astype(str)
)

if observed_oof_qids != expected_oof_qids:
    raise ValueError(
        "OOF predictions do not cover exactly the retrievable "
        "development queries"
    )

if (
    oof_scored.groupby("qid")["cv_fold"]
    .nunique()
    .ne(1)
    .any()
):
    raise ValueError(
        "An OOF query appears in multiple folds"
    )

cv_fold_metrics_df = (
    pd.DataFrame(cv_fixed_round_rows)
    .sort_values("cv_fold")
    .reset_index(drop=True)
)

(
    development_oof_combined,
    development_oof_conditional,
    development_oof_end_to_end,
    development_oof_query_details,
    development_oof_end_to_end_details,
) = evaluate_scored_predictions(
    oof_scored,
    "development",
    "Development four-fold OOF",
)

print("\nFixed-round CV metrics by fold:")
display(cv_fold_metrics_df)

print("\nPooled OOF development metrics:")
display(
    pd.DataFrame([
        development_oof_combined
    ])
)


# Final model: all 16 development sites

print(
    "\nTraining final model on all development websites "
    f"for {FINAL_NUM_BOOST_ROUND} rounds..."
)

model = fit_full_development_model(
    best_params,
    ALL_FEATURES,
    FINAL_NUM_BOOST_ROUND,
)

cv_round_selection_df.to_csv(
    RESULT_ROOT / "cv_round_selection.csv",
    index=False,
)
cv_fold_metrics_df.to_csv(
    RESULT_ROOT / "development_cv_fold_metrics.csv",
    index=False,
)
oof_scored.to_csv(
    ARTIFACT_ROOT / "development_oof_candidate_scores.csv",
    index=False,
)

print(
    "Final model current iteration:",
    model.current_iteration(),
)
print("Saved:", RESULT_ROOT / "cv_round_selection.csv")
print(
    "Saved:",
    RESULT_ROOT / "development_cv_fold_metrics.csv",
)
print(
    "Saved:",
    ARTIFACT_ROOT / "development_oof_candidate_scores.csv",
)


# Gain-based feature importance

gain_importance = model.feature_importance(
    importance_type="gain"
)

if len(gain_importance) != len(ALL_FEATURES):
    raise ValueError(
        "Feature-importance length does not match the "
        "50-feature model vector"
    )

feature_groups = {
    **{feature: "lexical" for feature in LEXICAL_FEATURES},
    **{feature: "semantic" for feature in SEMANTIC_FEATURES},
    **{feature: "structural" for feature in STRUCTURAL_FEATURES},
    **{feature: "attribute" for feature in ATTRIBUTE_FEATURES},
    **{feature: "geometry" for feature in GEOMETRY_FEATURES},
}

feature_importance_df = pd.DataFrame({
    "vector_index": np.arange(len(ALL_FEATURES)),
    "feature": ALL_FEATURES,
    "group": [
        feature_groups[feature]
        for feature in ALL_FEATURES
    ],
    "importance_gain": gain_importance,
})

total_gain = float(
    feature_importance_df["importance_gain"].sum()
)
feature_importance_df["importance_gain_pct"] = np.where(
    total_gain > 0,
    (
        feature_importance_df["importance_gain"]
        / total_gain
        * 100.0
    ),
    0.0,
)

feature_importance_sorted_df = (
    feature_importance_df
    .sort_values(
        "importance_gain",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

grouped_feature_importance_df = (
    feature_importance_df
    .groupby("group", as_index=False)
    .agg(
        importance_gain=("importance_gain", "sum"),
        n_features=("feature", "count"),
    )
    .sort_values(
        "importance_gain",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

group_total_gain = float(
    grouped_feature_importance_df[
        "importance_gain"
    ].sum()
)
grouped_feature_importance_df[
    "importance_gain_pct"
] = np.where(
    group_total_gain > 0,
    (
        grouped_feature_importance_df[
            "importance_gain"
        ]
        / group_total_gain
        * 100.0
    ),
    0.0,
)

print("\nTop 20 features by gain:")
display(
    feature_importance_sorted_df.head(20)
)

print("\nGrouped feature importance by gain:")
display(grouped_feature_importance_df)

feature_importance_df.to_csv(
    RESULT_ROOT / "feature_importance_vector_order.csv",
    index=False,
)
feature_importance_sorted_df.to_csv(
    RESULT_ROOT / "feature_importance_sorted.csv",
    index=False,
)
grouped_feature_importance_df.to_csv(
    RESULT_ROOT / "feature_importance_grouped.csv",
    index=False,
)

print("\nSaved gain-based feature importance:")
print(
    " ",
    RESULT_ROOT / "feature_importance_vector_order.csv",
)
print(
    " ",
    RESULT_ROOT / "feature_importance_sorted.csv",
)
print(
    " ",
    RESULT_ROOT / "feature_importance_grouped.csv",
)


## 8. Evaluate development OOF and held-out test predictions


In [ ]:
def score_model_on_frame(
    model_local,
    frame,
    features,
    *,
    num_iteration,
):
    ordered = sort_ranking_rows(frame)

    scored = ordered[
        ["qid", "site", "cv_fold", "label", "relevance"]
    ].copy().reset_index(drop=True)

    scored["score"] = model_local.predict(
        ordered[features].astype(np.float32),
        num_iteration=int(num_iteration),
    )

    return scored


test_scored = score_model_on_frame(
    model,
    test_df,
    ALL_FEATURES,
    num_iteration=FINAL_NUM_BOOST_ROUND,
)

(
    test_combined,
    test_conditional,
    test_end_to_end,
    test_query_details,
    test_end_to_end_details,
) = evaluate_scored_predictions(
    test_scored,
    "test",
    "Held-out test",
)

evaluation_summary_df = pd.DataFrame([
    development_oof_combined,
    test_combined,
])

conditional_summary_df = pd.DataFrame([
    {
        "name": "Development four-fold OOF",
        **development_oof_conditional,
    },
    {
        "name": "Held-out test",
        **test_conditional,
    },
])

end_to_end_summary_df = pd.DataFrame([
    development_oof_end_to_end,
    test_end_to_end,
])

metric_columns_for_cv = [
    "conditional_top1",
    "conditional_mrr",
    "conditional_hit3",
    "conditional_hit5",
    "conditional_ndcg1",
    "conditional_ndcg3",
    "conditional_ndcg5",
    "candidate_recall",
    "end_to_end_top1",
    "end_to_end_mrr",
    "end_to_end_hit3",
    "end_to_end_hit5",
    "end_to_end_ndcg1",
    "end_to_end_ndcg3",
    "end_to_end_ndcg5",
]

cv_mean_std_rows = []

for metric in metric_columns_for_cv:
    values = cv_fold_metrics_df[
        metric
    ].to_numpy(dtype=float)

    cv_mean_std_rows.append({
        "metric": metric,
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=0)),
    })

cv_mean_std_df = pd.DataFrame(cv_mean_std_rows)

print(
    "Combined conditional and end-to-end ranking metrics:"
)
display(evaluation_summary_df)

print(
    "Development fixed-round metrics by CV fold:"
)
display(cv_fold_metrics_df)

print(
    "Development four-fold mean ± standard deviation:"
)
display(cv_mean_std_df)

evaluation_summary_df.to_csv(
    RESULT_ROOT / "ranking_metrics_combined.csv",
    index=False,
)
conditional_summary_df.to_csv(
    RESULT_ROOT / "conditional_ranking_metrics.csv",
    index=False,
)
end_to_end_summary_df.to_csv(
    RESULT_ROOT / "end_to_end_ranking_metrics.csv",
    index=False,
)
cv_mean_std_df.to_csv(
    RESULT_ROOT / "development_cv_mean_std.csv",
    index=False,
)
development_oof_end_to_end_details.to_csv(
    RESULT_ROOT / "development_oof_query_ranks_end_to_end.csv",
    index=False,
)
test_end_to_end_details.to_csv(
    RESULT_ROOT / "test_query_ranks_end_to_end.csv",
    index=False,
)
test_scored.to_csv(
    ARTIFACT_ROOT / "test_candidate_scores.csv",
    index=False,
)

print(
    "Saved combined cross-validation and test outputs to",
    RESULT_ROOT,
)


## 9. Save the model and base runtime configuration

This notebook saves `config_base.pkl`. REPATH_03 adds the calibrated confidence threshold and writes the final runtime `config.pkl`.


In [ ]:
model.save_model(
    str(MODEL_ROOT / "lgb_ranker.txt")
)

saved_config_base = {
    "ALL_FEATURES": ALL_FEATURES,
    "LEXICAL_FEATURES": LEXICAL_FEATURES,
    "SEMANTIC_FEATURES": SEMANTIC_FEATURES,
    "STRUCTURAL_FEATURES": STRUCTURAL_FEATURES,
    "ATTRIBUTE_FEATURES": ATTRIBUTE_FEATURES,
    "GEOMETRY_FEATURES": GEOMETRY_FEATURES,
    "feature_group_names": [
        "lexical",
        "semantic",
        "structural",
        "attribute",
        "geometry",
    ],
    "best_params": best_params,
    "num_boost_round": int(
        FINAL_NUM_BOOST_ROUND
    ),
    "cv_folds": int(CV_FOLDS),
    "development_site_to_fold": {
        site: int(fold)
        for site, fold in site_to_fold.items()
    },
    "cv_round_selection": (
        cv_round_selection_df.to_dict(
            orient="records"
        )
    ),
    "development_cv_fold_metrics": (
        cv_fold_metrics_df.to_dict(
            orient="records"
        )
    ),
    "development_cv_mean_std": (
        cv_mean_std_df.to_dict(
            orient="records"
        )
    ),
    "development_oof_combined_metrics": (
        development_oof_combined
    ),
    "test_combined_metrics": test_combined,
    "development_oof_conditional_metrics": (
        development_oof_conditional
    ),
    "test_conditional_metrics": (
        test_conditional
    ),
    "development_oof_end_to_end_metrics": (
        development_oof_end_to_end
    ),
    "test_end_to_end_metrics": (
        test_end_to_end
    ),
    "sbert_model": SBERT_MODEL_NAME,
    "training_input": "csv_only",
    "selection_protocol": (
        "four_fold_site_grouped_cross_validation"
    ),
    "final_training_sites": 16,
    "held_out_test_sites": 6,
    "pair_csv_name": PAIR_CSV.name,
    "query_manifest_csv_name": (
        QUERY_MANIFEST_CSV.name
    ),
    "feature_schema_csv_name": (
        FEATURE_SCHEMA_CSV.name
    ),
}

joblib.dump(
    saved_config_base,
    MODEL_ROOT / "config_base.pkl",
)

feature_schema_df.to_csv(
    RESULT_ROOT / "feature_schema_used.csv",
    index=False,
)

print("Saved base model artifacts:")
print(" ", MODEL_ROOT / "lgb_ranker.txt")
print(" ", MODEL_ROOT / "config_base.pkl")
print(" ", RESULT_ROOT / "feature_schema_used.csv")
print("Features        :", len(ALL_FEATURES))
print("CV folds        :", CV_FOLDS)
print(
    "Boosting rounds :",
    FINAL_NUM_BOOST_ROUND,
)
print(
    "Run REPATH 03 after this notebook to calibrate "
    "the tau margin and create final config.pkl."
)


## 10. Report held-out performance by site

Each site row reports both conditional and end-to-end metrics.


In [ ]:
site_rows = []

for site in sorted(
    manifest_df.loc[
        manifest_df["split"] == "test",
        "site",
    ].unique()
):
    site_scored = test_scored[
        test_scored["site"] == site
    ].copy()

    site_combined, _, _, _, _ = (
        evaluate_scored_predictions(
            site_scored,
            "test",
            site,
            site=site,
        )
    )

    site_rows.append(site_combined)

test_metrics_by_site = (
    pd.DataFrame(site_rows)
    .sort_values(
        [
            "end_to_end_top1",
            "conditional_top1",
            "end_to_end_mrr",
            "end_to_end_ndcg5",
        ],
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

display(test_metrics_by_site)

test_metrics_by_site.to_csv(
    RESULT_ROOT / "test_metrics_by_site_combined.csv",
    index=False,
)

print(
    "Saved:",
    RESULT_ROOT / "test_metrics_by_site_combined.csv",
)


## 11. Evaluate feature-group combinations

The five feature groups produce 31 non-empty combinations. For each reduced combination:

1. the full-model hyperparameters remain frozen;
2. four development folds select a median boosting-round count;
3. four fixed-round fold models generate OOF development predictions;
4. one model is trained on all development websites;
5. both conditional and end-to-end metrics are calculated on development OOF and held-out test queries.

The final table is sorted by held-out end-to-end Top-1, followed by conditional Top-1 and NDCG@5. This sorting is only for presentation.


In [ ]:
# Exhaustive feature-group combination ablation

FEATURE_GROUPS = {
    "lexical": list(LEXICAL_FEATURES),
    "semantic": list(SEMANTIC_FEATURES),
    "structural": list(STRUCTURAL_FEATURES),
    "attribute": list(ATTRIBUTE_FEATURES),
    "geometry": list(GEOMETRY_FEATURES),
}

GROUP_ORDER = list(FEATURE_GROUPS)
GROUP_SETS = {
    group: set(features)
    for group, features in FEATURE_GROUPS.items()
}

combination_specs = []

for n_groups in range(len(GROUP_ORDER), 0, -1):
    for included_groups in combinations(
        GROUP_ORDER,
        n_groups,
    ):
        included_groups = tuple(included_groups)
        excluded_groups = tuple(
            group
            for group in GROUP_ORDER
            if group not in included_groups
        )

        selected_features = [
            feature
            for feature in ALL_FEATURES
            if any(
                feature in GROUP_SETS[group]
                for group in included_groups
            )
        ]

        combination_specs.append({
            "included_groups_tuple": included_groups,
            "included_groups": " + ".join(
                included_groups
            ),
            "excluded_groups": (
                " + ".join(excluded_groups)
                if excluded_groups
                else "none"
            ),
            "n_groups": int(n_groups),
            "n_features": int(
                len(selected_features)
            ),
            "features": selected_features,
        })

if len(combination_specs) != 31:
    raise ValueError(
        "Five feature groups must produce exactly "
        "31 non-empty combinations"
    )

for spec in combination_specs:
    features = spec["features"]

    if not features:
        raise ValueError(
            f"{spec['included_groups']}: empty feature set"
        )

    if len(features) != len(set(features)):
        raise ValueError(
            f"{spec['included_groups']}: duplicate features"
        )

    unknown = sorted(
        set(features) - set(ALL_FEATURES)
    )
    if unknown:
        raise ValueError(
            f"{spec['included_groups']}: "
            f"unknown features {unknown}"
        )


print(
    "Running 31 feature-group combinations."
)
print(
    "The full model reuses the main run. Each of the other "
    "30 combinations performs four early-stopping fits, "
    "four fixed-round OOF fits, and one final development fit."
)
print(
    "This section is intentionally computationally expensive."
)

ablation_rows = []
ablation_round_rows = []
ablation_fold_rows = []
ablation_query_detail_parts = []

for combination_index, spec in enumerate(
    tqdm(
        combination_specs,
        desc="Feature-group combinations",
    )
):
    included_groups = spec[
        "included_groups_tuple"
    ]
    included_label = spec["included_groups"]
    features = spec["features"]
    is_full_model = (
        len(included_groups) == len(GROUP_ORDER)
    )

    print(
        f"\nCombination {combination_index + 1}/"
        f"{len(combination_specs)}: "
        f"{included_label}"
    )
    print("Features:", len(features))

    if is_full_model:
        selected_rounds = int(
            FINAL_NUM_BOOST_ROUND
        )
        fold_iterations = (
            cv_round_selection_df[
                "best_iteration"
            ]
            .astype(int)
            .tolist()
        )
        variant_oof_scored = (
            oof_scored.copy()
        )
        variant_model = model
        variant_test_scored = (
            test_scored.copy()
        )

        for _, fold_row in (
            cv_round_selection_df.iterrows()
        ):
            ablation_round_rows.append({
                "combination_index": (
                    combination_index
                ),
                "included_groups": (
                    included_label
                ),
                "cv_fold": int(
                    fold_row["cv_fold"]
                ),
                "best_iteration": int(
                    fold_row["best_iteration"]
                ),
                "selected_median_rounds": (
                    selected_rounds
                ),
                **{
                    f"early_stopping_{column}": (
                        fold_row[column]
                    )
                    for column in (
                        cv_round_selection_df.columns
                    )
                    if column not in {
                        "cv_fold",
                        "best_iteration",
                    }
                },
            })

        for _, fold_row in (
            cv_fold_metrics_df.iterrows()
        ):
            ablation_fold_rows.append({
                "combination_index": (
                    combination_index
                ),
                "included_groups": (
                    included_label
                ),
                **fold_row.to_dict(),
            })

    else:
        fold_iterations = []

        for fold in range(CV_FOLDS):
            (
                fold_model_es,
                fold_metrics_es,
                _,
            ) = fit_fold_with_early_stopping(
                best_params,
                fold,
                features,
                max_rounds=(
                    FINAL_MAX_BOOST_ROUNDS
                ),
            )

            best_iteration = (
                effective_iteration(
                    fold_model_es
                )
            )
            fold_iterations.append(
                best_iteration
            )

            ablation_round_rows.append({
                "combination_index": (
                    combination_index
                ),
                "included_groups": (
                    included_label
                ),
                "cv_fold": fold,
                "best_iteration": int(
                    best_iteration
                ),
                **{
                    f"early_stopping_{key}": value
                    for key, value in (
                        fold_metrics_es.items()
                    )
                    if key != "name"
                },
            })

        selected_rounds = max(
            1,
            int(round(
                float(np.median(
                    fold_iterations
                ))
            )),
        )

        for row in ablation_round_rows:
            if (
                row["combination_index"]
                == combination_index
            ):
                row[
                    "selected_median_rounds"
                ] = selected_rounds

        variant_oof_parts = []

        for fold in range(CV_FOLDS):
            (
                _,
                fold_metrics_fixed,
                fold_scored_fixed,
            ) = fit_fold_fixed_rounds(
                best_params,
                fold,
                features,
                selected_rounds,
            )

            variant_oof_parts.append(
                fold_scored_fixed
            )

            ablation_fold_rows.append({
                "combination_index": (
                    combination_index
                ),
                "included_groups": (
                    included_label
                ),
                "cv_fold": fold,
                "n_features": len(features),
                "num_boost_round": (
                    selected_rounds
                ),
                **{
                    key: value
                    for key, value in (
                        fold_metrics_fixed.items()
                    )
                    if key != "name"
                },
            })

        variant_oof_scored = (
            pd.concat(
                variant_oof_parts,
                ignore_index=True,
            )
            .sort_values(
                ["qid", "score"],
                ascending=[True, False],
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        variant_model = (
            fit_full_development_model(
                best_params,
                features,
                selected_rounds,
            )
        )

        variant_test_scored = (
            score_model_on_frame(
                variant_model,
                test_df,
                features,
                num_iteration=(
                    selected_rounds
                ),
            )
        )

    (
        development_combined,
        _,
        _,
        development_conditional_details,
        development_end_to_end_details,
    ) = evaluate_scored_predictions(
        variant_oof_scored,
        "development",
        (
            f"{included_label} "
            "development OOF"
        ),
    )

    (
        test_variant_combined,
        _,
        _,
        test_conditional_details_variant,
        test_end_to_end_details_variant,
    ) = evaluate_scored_predictions(
        variant_test_scored,
        "test",
        f"{included_label} held-out test",
    )

    row = {
        "combination_index": combination_index,
        "included_groups": included_label,
        "excluded_groups": (
            spec["excluded_groups"]
        ),
        "n_groups": spec["n_groups"],
        "n_features": spec["n_features"],
        "selected_cv_median_rounds": int(
            selected_rounds
        ),
        "cv_min_best_iteration": int(
            np.min(fold_iterations)
        ),
        "cv_max_best_iteration": int(
            np.max(fold_iterations)
        ),
    }

    row.update({
        f"development_{key}": value
        for key, value in (
            development_combined.items()
        )
        if key != "name"
    })

    row.update({
        f"test_{key}": value
        for key, value in (
            test_variant_combined.items()
        )
        if key != "name"
    })

    ablation_rows.append(row)

    for split_name, details in [
        (
            "development",
            development_end_to_end_details,
        ),
        (
            "test",
            test_end_to_end_details_variant,
        ),
    ]:
        detail_copy = details.copy()
        detail_copy.insert(
            0,
            "split_evaluated",
            split_name,
        )
        detail_copy.insert(
            0,
            "included_groups",
            included_label,
        )
        detail_copy.insert(
            0,
            "combination_index",
            combination_index,
        )
        ablation_query_detail_parts.append(
            detail_copy
        )

    print(
        "Development conditional Top-1:",
        f"{development_combined['conditional_top1']:.4f}",
    )
    print(
        "Development end-to-end Top-1:",
        f"{development_combined['end_to_end_top1']:.4f}",
    )
    print(
        "Test conditional Top-1:",
        f"{test_variant_combined['conditional_top1']:.4f}",
    )
    print(
        "Test end-to-end Top-1:",
        f"{test_variant_combined['end_to_end_top1']:.4f}",
    )


ablation_df = pd.DataFrame(ablation_rows)

full_mask = (
    ablation_df["n_groups"]
    == len(GROUP_ORDER)
)

if int(full_mask.sum()) != 1:
    raise ValueError(
        "Exactly one full-model combination is required"
    )

full_row = ablation_df.loc[
    full_mask
].iloc[0]

delta_metrics = [
    "conditional_top1",
    "conditional_mrr",
    "conditional_ndcg5",
    "end_to_end_top1",
    "end_to_end_mrr",
    "end_to_end_ndcg5",
]

for split_name in [
    "development",
    "test",
]:
    for metric in delta_metrics:
        source_column = (
            f"{split_name}_{metric}"
        )
        delta_column = (
            f"delta_{source_column}_vs_full"
        )
        ablation_df[delta_column] = (
            ablation_df[source_column]
            - full_row[source_column]
        )

ablation_df = (
    ablation_df
    .sort_values(
        [
            "test_end_to_end_top1",
            "test_conditional_top1",
            "test_end_to_end_ndcg5",
            "n_groups",
            "included_groups",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

ablation_df.insert(
    0,
    "performance_rank",
    np.arange(
        1,
        len(ablation_df) + 1,
    ),
)

ablation_rounds_df = (
    pd.DataFrame(ablation_round_rows)
    .sort_values(
        [
            "combination_index",
            "cv_fold",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

ablation_fold_metrics_df = (
    pd.DataFrame(ablation_fold_rows)
    .sort_values(
        [
            "combination_index",
            "cv_fold",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

ablation_query_details_df = (
    pd.concat(
        ablation_query_detail_parts,
        ignore_index=True,
    )
)

paper_columns = [
    "performance_rank",
    "included_groups",
    "excluded_groups",
    "n_groups",
    "n_features",
    "selected_cv_median_rounds",
    "development_conditional_top1",
    "development_end_to_end_top1",
    "development_conditional_mrr",
    "development_end_to_end_mrr",
    "development_conditional_ndcg5",
    "development_end_to_end_ndcg5",
    "test_conditional_top1",
    "test_end_to_end_top1",
    "test_conditional_mrr",
    "test_end_to_end_mrr",
    "test_conditional_ndcg5",
    "test_end_to_end_ndcg5",
    "delta_test_conditional_top1_vs_full",
    "delta_test_end_to_end_top1_vs_full",
    "delta_test_conditional_ndcg5_vs_full",
    "delta_test_end_to_end_ndcg5_vs_full",
]

ablation_sorted_summary_df = (
    ablation_df[paper_columns].copy()
)

print(
    "\nAll 31 feature-group combinations, "
    "sorted by held-out end-to-end Top-1:"
)
display(ablation_sorted_summary_df)

ablation_df.to_csv(
    RESULT_ROOT / "feature_group_all_combinations_full.csv",
    index=False,
)
ablation_sorted_summary_df.to_csv(
    RESULT_ROOT / "feature_group_all_combinations_sorted.csv",
    index=False,
)
ablation_rounds_df.to_csv(
    RESULT_ROOT / "feature_group_all_combinations_cv_rounds.csv",
    index=False,
)
ablation_fold_metrics_df.to_csv(
    RESULT_ROOT / "feature_group_all_combinations_fold_metrics.csv",
    index=False,
)
ablation_query_details_df.to_csv(
    RESULT_ROOT / "feature_group_all_combinations_query_details.csv",
    index=False,
)

print("\nSaved exhaustive ablation outputs:")
print(
    " ",
    RESULT_ROOT / "feature_group_all_combinations_sorted.csv",
)
print(
    " ",
    RESULT_ROOT / "feature_group_all_combinations_full.csv",
)
print(
    " ",
    RESULT_ROOT / "feature_group_all_combinations_cv_rounds.csv",
)
print(
    " ",
    RESULT_ROOT / "feature_group_all_combinations_fold_metrics.csv",
)
print(
    " ",
    RESULT_ROOT / "feature_group_all_combinations_query_details.csv",
)

print(
    "\nAblation results are post-hoc analyses. "
    "They are not used to change the split, "
    "hyperparameters, feature groups, or final model."
)


## Reference output

The bundled artifacts correspond to the following ranking results:

| Split | Framework | Retrievable | Candidate recall | Conditional Top-1 | End-to-end Top-1 | End-to-end MRR | End-to-end NDCG@5 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Development four-fold OOF | 782 | 781 | 99.87% | 98.08% | 97.95% | 98.64% | 98.88% |
| Held-out test | 318 | 314 | 98.74% | 96.18% | 94.97% | 96.38% | 96.77% |

The final model and base configuration are written to `models/`. The pooled development OOF and held-out test candidate scores required for calibration are written to `artifacts/repath/`. Detailed training and ablation reports are written to `results/notebooks/training/`.
